# Feature Engineering — Building Better Inputs for Your Model

## Why Feature Engineering Matters

> *"Applied machine learning is basically feature engineering."* — Andrew Ng

A mediocre model on excellent features beats an excellent model on raw features.

Feature engineering is how you:
- Inject **domain knowledge** into the model
- Fix **non-linearities** that linear models can't learn
- Create **interaction effects** that individual features miss
- Handle **skew and scale** that hurt gradient-based optimization
- Encode **categorical meaning** that strings can't convey

## This Notebook

Implements every decision made in `eda_visualization.ipynb` — the EDA report
becomes the engineering spec.

| Section | Engineering Technique |
|---|---|
| 1 | Numeric transforms (log, polynomial, clipping) |
| 2 | Interaction features (multiply, divide, combine) |
| 3 | Categorical encoding (ordinal, one-hot, target encoding) |
| 4 | GroupBy features (neighborhood statistics, target means) |
| 5 | Domain knowledge features (house-specific) |
| 6 | Feature selection (variance, correlation, model-based) |
| 7 | The full sklearn Pipeline |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats as scipy_stats
from pathlib import Path

np.random.seed(42)
pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 110)
pd.set_option('display.float_format', '{:.3f}'.format)
sns.set_theme(style='whitegrid', palette='muted', font_scale=0.95)

# ── Synthetic House Prices dataset (Kaggle-style) ───────────────────────────
n = 600

neighborhoods = ['NAmes','CollgCr','OldTown','Edwards','Somerst',
                 'NridgHt','Gilbert','Sawyer','BrkSide','Crawfor']
bldg_types    = ['1Fam','TwnhsE','Duplex','2fmCon']
conditions    = ['Excellent','Good','Typical','Fair','Poor']

qual    = np.random.randint(3, 11, n)
area    = np.random.normal(1500, 420, n).clip(400, 5000).round(0)
yr_built = np.random.randint(1930, 2023, n)
garage  = np.random.choice([0,1,2,3], n, p=[0.05,0.18,0.62,0.15]).astype(float)
bsmt    = np.random.normal(950, 320, n).clip(0, 3000).round(0)
lot     = np.random.lognormal(9.0, 0.5, n).clip(1500, 50000).round(0)
rooms   = np.random.randint(4, 12, n)
full_bath = np.random.choice([1,2,3], n, p=[0.30,0.60,0.10])
nbhd    = np.random.choice(neighborhoods, n,
          p=[0.13,0.13,0.09,0.09,0.11,0.11,0.11,0.09,0.07,0.07])
btype   = np.random.choice(bldg_types, n, p=[0.72,0.16,0.06,0.06])
cond    = np.random.choice(conditions, n, p=[0.10,0.30,0.40,0.15,0.05])

nbhd_premium = {'NAmes':0,'CollgCr':5000,'OldTown':-15000,'Edwards':-20000,
                'Somerst':15000,'NridgHt':40000,'Gilbert':10000,'Sawyer':-5000,
                'BrkSide':-25000,'Crawfor':8000}
cond_mult = {'Excellent':1.15,'Good':1.05,'Typical':1.0,'Fair':0.93,'Poor':0.85}

price = (
    area * 82 +
    qual * 9500 +
    (yr_built - 1930) * 420 +
    garage * 5500 +
    bsmt * 28 +
    np.log1p(lot) * 2500 +
    rooms * 1800 +
    full_bath * 7000 +
    np.array([nbhd_premium[x] for x in nbhd]) +
    np.random.normal(0, 18000, n)
)
price = price * np.array([cond_mult[x] for x in cond])
price = price.clip(55000, 780000).round(-3)

# Inject realistic missing values
miss_idx = np.random.permutation(n)
garage_na = miss_idx[:35]
bsmt_na   = miss_idx[35:55]

df = pd.DataFrame({
    'Id':          np.arange(1, n+1),
    'GrLivArea':   area,
    'OverallQual': qual,
    'YearBuilt':   yr_built,
    'GarageCars':  garage,
    'TotalBsmtSF': bsmt,
    'LotArea':     lot,
    'TotRmsAbvGrd':rooms,
    'FullBath':    full_bath,
    'Neighborhood':nbhd,
    'BldgType':    btype,
    'Condition':   cond,
    'SalePrice':   price,
})
df.loc[garage_na, 'GarageCars'] = np.nan
df.loc[bsmt_na,   'TotalBsmtSF'] = np.nan

# Clean for use (impute, don't need missing in most sections)
df_clean = df.copy()
df_clean['GarageCars'] = df_clean['GarageCars'].fillna(0)
df_clean['TotalBsmtSF'] = df_clean['TotalBsmtSF'].fillna(0)
df_clean['HouseAge']    = 2024 - df_clean['YearBuilt']
df_clean['LogPrice']    = np.log1p(df_clean['SalePrice'])
df_clean['log_SalePrice'] = df_clean['LogPrice']

NUMERIC_COLS = ['GrLivArea','OverallQual','YearBuilt','GarageCars',
                'TotalBsmtSF','LotArea','TotRmsAbvGrd','FullBath','SalePrice']
CAT_COLS     = ['Neighborhood','BldgType','Condition']
TARGET       = 'SalePrice'

print(f"Dataset: {df.shape[0]} rows x {df.shape[1]} cols")
print(f"Numeric: {len(NUMERIC_COLS)}   Categorical: {len(CAT_COLS)}   Target: {TARGET}")
print(f"Price range: ${df_clean[TARGET].min():,.0f} - ${df_clean[TARGET].max():,.0f}")


---
## Section 1 — Numeric Transforms

### Three Transforms You'll Use Constantly

**Log transform** — right-skewed distributions, prices, areas, counts
```python
np.log1p(x)    # log(x + 1) — safe for zeros
np.expm1(y)    # inverse: e^y - 1
```

**Polynomial features** — capture non-linear relationships
```python
X['qual_sq'] = X['OverallQual'] ** 2    # manual
# Or: PolynomialFeatures(degree=2, include_bias=False)
```

**Winsorization (clipping)** — tame outliers without removing rows
```python
lo, hi = df['col'].quantile([0.01, 0.99])
df['col_clip'] = df['col'].clip(lo, hi)
```

### When to Apply Each

| Situation | Transform |
|---|---|
| Right-skewed (long tail right) | log1p |
| Left-skewed (long tail left) | square or cube |
| Accelerating relationship with target | polynomial |
| Outliers (data entry errors) | clip/winsorize |
| Different units across features | StandardScaler (in pipeline) |


In [ ]:
print("=== Numeric Transforms ===")
print()

df_fe = df_clean.copy()

# ── Log transforms ────────────────────────────────────────────────────────────
# EDA finding: LotArea and TotalBsmtSF are right-skewed
df_fe['log_LotArea']    = np.log1p(df_fe['LotArea'])
df_fe['log_TotalBsmtSF']= np.log1p(df_fe['TotalBsmtSF'])
# SalePrice: always predict log-transformed, exponentiate back
df_fe['log_SalePrice']  = np.log1p(df_fe['SalePrice'])

print("Log transform effect on skewness:")
for col, log_col in [('LotArea','log_LotArea'),
                      ('TotalBsmtSF','log_TotalBsmtSF'),
                      ('SalePrice','log_SalePrice')]:
    raw_skew = df_fe[col].skew()
    log_skew = df_fe[log_col].skew()
    print(f"  {col:15}: skew {raw_skew:+.2f} -> {log_skew:+.2f}   delta={log_skew-raw_skew:+.2f}")
print()

# ── Polynomial features ───────────────────────────────────────────────────────
# EDA finding: OverallQual has accelerating (near-exponential) relationship with price
df_fe['OverallQual_sq'] = df_fe['OverallQual'] ** 2
df_fe['HouseAge_sq']    = df_fe['HouseAge'] ** 2

# How much does the squared term add?
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

X_base = df_fe[['OverallQual','HouseAge']].values
X_poly = df_fe[['OverallQual','HouseAge','OverallQual_sq','HouseAge_sq']].values
y      = df_fe['log_SalePrice'].values

def cv_r2(X, y, alpha=1.0):
    sc    = StandardScaler()
    X_sc  = sc.fit_transform(X)
    model = Ridge(alpha=alpha)
    scores = cross_val_score(model, X_sc, y, cv=5, scoring='r2')
    return scores.mean(), scores.std()

r2_base, std_base = cv_r2(X_base, y)
r2_poly, std_poly = cv_r2(X_poly, y)
print(f"Polynomial features lift:")
print(f"  Base (Qual + Age):           CV R2 = {r2_base:.4f} +/- {std_base:.4f}")
print(f"  + Qual^2 + Age^2:            CV R2 = {r2_poly:.4f} +/- {std_poly:.4f}  (+{r2_poly-r2_base:+.4f})")
print()

# ── Winsorization ─────────────────────────────────────────────────────────────
# EDA finding: LotArea and GrLivArea have long tails (outliers)
print("Winsorization (clip at 1st-99th percentile):")
for col in ['LotArea', 'GrLivArea']:
    lo_p, hi_p   = df_fe[col].quantile(0.01), df_fe[col].quantile(0.99)
    n_clipped    = ((df_fe[col] < lo_p) | (df_fe[col] > hi_p)).sum()
    df_fe[col + '_clip'] = df_fe[col].clip(lo_p, hi_p)
    print(f"  {col:12}: [{lo_p:,.0f}, {hi_p:,.0f}]   clipped {n_clipped} values")
print()

# ── Visualization: before/after transforms ────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.suptitle("Numeric Transforms: Before vs After", fontsize=12, fontweight='bold')

pairs = [
    ('LotArea',    'log_LotArea',    'coral',      'steelblue'),
    ('TotalBsmtSF','log_TotalBsmtSF','coral',      'steelblue'),
    ('OverallQual','OverallQual_sq', 'goldenrod',   'darkgreen'),
    ('HouseAge',   'HouseAge_sq',   'mediumpurple','teal'),
]
for i, (orig, trans, c1, c2) in enumerate(pairs):
    axes[0,i].hist(df_fe[orig],  bins=30, color=c1, edgecolor='white', linewidth=0.4, alpha=0.85)
    axes[0,i].set_title(orig + "  skew=" + f"{df_fe[orig].skew():.2f}", fontsize=8)
    axes[1,i].hist(df_fe[trans], bins=30, color=c2, edgecolor='white', linewidth=0.4, alpha=0.85)
    axes[1,i].set_title(trans + "  skew=" + f"{df_fe[trans].skew():.2f}", fontsize=8)

for ax in axes.flatten():
    ax.tick_params(labelsize=7)

plt.tight_layout()
plt.savefig('/tmp/fe_transforms.png', dpi=100, bbox_inches='tight')
plt.show()


---
## Section 2 — Interaction Features

### What Interactions Capture

A linear model assumes features contribute **independently** to the target:

```
price = w1*area + w2*quality + w3*age + bias
```

But in reality: a 2000 sqft home in **excellent condition** is worth much more
than the sum of (2000 sqft bonus) + (excellent condition bonus) separately.
Size and quality **interact** — the combination is more than the parts.

**Interaction feature:**
```python
df['area_x_qual'] = df['GrLivArea'] * df['OverallQual']
```

Now the model can learn: `w * area_x_qual` captures the joint effect.

### Types of Interactions

| Type | Formula | Intuition |
|---|---|---|
| Multiplicative | `a * b` | "more of both is exponentially better" |
| Ratio | `a / b` | "efficiency" — value per unit |
| Difference | `a - b` | "gap" — above/below baseline |
| Indicator | `(a > threshold).astype(int)` | "step change at a level" |


In [ ]:
print("=== Interaction Features ===")
print()

# ── Multiplicative interactions ───────────────────────────────────────────────
# EDA: GrLivArea and OverallQual both have strong linear correlation
# Together they should interact
df_fe['area_x_qual']    = df_fe['GrLivArea']    * df_fe['OverallQual']
df_fe['bsmt_x_qual']    = df_fe['TotalBsmtSF']  * df_fe['OverallQual']
df_fe['garage_x_qual']  = df_fe['GarageCars']   * df_fe['OverallQual']

# ── Ratio features (efficiency) ───────────────────────────────────────────────
# avg room size: large area with few rooms = spacious
df_fe['avg_room_size']  = df_fe['GrLivArea'] / (df_fe['TotRmsAbvGrd'] + 1)  # +1 avoids div-by-zero
# total indoor space
df_fe['total_sf']       = df_fe['GrLivArea'] + df_fe['TotalBsmtSF']
# garage coverage ratio
df_fe['garage_ratio']   = df_fe['GarageCars'] / (df_fe['TotRmsAbvGrd'] + 1)

# ── Binary threshold indicators ───────────────────────────────────────────────
# EDA: has_garage and has_basement are distinct states (not just 0)
df_fe['has_garage']     = (df_fe['GarageCars']   > 0).astype(int)
df_fe['has_basement']   = (df_fe['TotalBsmtSF']  > 0).astype(int)
df_fe['is_new']         = (df_fe['HouseAge']      < 15).astype(int)
df_fe['is_large']       = (df_fe['GrLivArea']     > 2000).astype(int)
df_fe['is_high_qual']   = (df_fe['OverallQual']   >= 8).astype(int)

# ── Measure lift from each interaction ───────────────────────────────────────
print("Correlation lift: do interaction features correlate more with log(SalePrice)?")
print()

base_feats = ['GrLivArea','OverallQual','TotalBsmtSF','GarageCars','TotRmsAbvGrd']
new_feats  = ['area_x_qual','bsmt_x_qual','total_sf','avg_room_size',
              'has_garage','has_basement','is_new','is_high_qual']

y_log = df_fe['log_SalePrice']
print(f"  {'Feature':20s}  {'r with log(SalePrice)':>22s}  Note")
print("  " + "-"*60)
for feat in base_feats:
    r = df_fe[feat].corr(y_log)
    print(f"  {feat:20s}  {r:>+22.4f}  (base)")
print()
for feat in new_feats:
    r = df_fe[feat].corr(y_log)
    # Find best base comparison
    best_base = max(base_feats, key=lambda f: abs(df_fe[f].corr(y_log)))
    r_base    = df_fe[best_base].corr(y_log)
    delta     = abs(r) - abs(r_base)
    flag      = " <- IMPROVED" if delta > 0.02 else ""
    print(f"  {feat:20s}  {r:>+22.4f}{flag}")
print()

# Full model comparison
base_X  = df_fe[base_feats].values
full_X  = df_fe[base_feats + new_feats].values
y_arr   = y_log.values

from sklearn.model_selection import cross_val_score
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

def cv_r2(X, y, alpha=10.0):
    sc = StandardScaler()
    X_sc = sc.fit_transform(X)
    return cross_val_score(Ridge(alpha=alpha), X_sc, y, cv=5, scoring='r2').mean()

r2_base_model = cv_r2(base_X,  y_arr)
r2_full_model = cv_r2(full_X,  y_arr)
print(f"Ridge CV R2 with base features only:       {r2_base_model:.4f}")
print(f"Ridge CV R2 with base + interactions:      {r2_full_model:.4f}  (+{r2_full_model-r2_base_model:.4f})")


---
## Section 3 — Categorical Encoding

### The Options

| Method | When to Use | Leakage Risk |
|---|---|---|
| **One-hot** | Low cardinality (<10 categories), no ordinal meaning | None |
| **Ordinal** | Natural order exists (Poor < Fair < Good < Excellent) | None |
| **Target encoding** | High cardinality (10+ categories), strong signal | YES — must use cross-fitting |
| **Frequency encoding** | Very high cardinality, when count is a proxy for signal | None |
| **Binary encoding** | High cardinality, want fewer columns than one-hot | None |

### Target Encoding — Handle with Care

Target encoding replaces each category with the mean of the target in that category.

**The leakage problem:**
```python
# WRONG — using the row's own target to compute its encoding
df['nbhd_encoded'] = df.groupby('Neighborhood')['SalePrice'].transform('mean')
```

**The correct approach — out-of-fold target encoding:**
Split training data into folds. For each row, compute the neighborhood mean
using all **other** folds (never the row's own fold). This is exactly what
`sklearn`'s `TargetEncoder` does.


In [ ]:
from sklearn.preprocessing import OrdinalEncoder, TargetEncoder
from sklearn.model_selection import KFold
import warnings; warnings.filterwarnings('ignore')

print("=== Categorical Encoding ===")
print()

# ── 1. One-hot encoding ───────────────────────────────────────────────────────
print("1. One-Hot Encoding (BldgType: 4 categories -> 3 columns)")
ohe_cols = pd.get_dummies(df_fe['BldgType'], prefix='BldgType', drop_first=True).astype(int)
print(ohe_cols.head(5))
df_fe = pd.concat([df_fe, ohe_cols], axis=1)
print(f"   Added {ohe_cols.shape[1]} one-hot columns for BldgType")
print()

# ── 2. Ordinal encoding ───────────────────────────────────────────────────────
print("2. Ordinal Encoding (Condition: natural order)")
condition_order = ['Poor','Fair','Typical','Good','Excellent']
cond_map        = {c: i+1 for i, c in enumerate(condition_order)}
df_fe['Condition_ord'] = df_fe['Condition'].map(cond_map)
print("   Mapping:", cond_map)
print("   Sample:")
print(df_fe[['Condition','Condition_ord']].drop_duplicates().sort_values('Condition_ord').to_string())
print()

# ── 3. Target encoding (Neighborhood — 10 categories) ────────────────────────
print("3. Target Encoding (Neighborhood: 10 categories)")
print()
print("   Naive (WRONG — leaks target into features):")
naive_enc = df_fe.groupby('Neighborhood')['log_SalePrice'].transform('mean')
r_naive   = naive_enc.corr(df_fe['log_SalePrice'])
print(f"   Naive correlation with log(SalePrice): {r_naive:.4f} <- inflated by leakage!")
print()

print("   Correct: cross-fold target encoding")

# Manual 5-fold cross-target-encoding (shows the concept)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
target_enc = np.zeros(len(df_fe))
y_arr      = df_fe['log_SalePrice'].values
nbhd_arr   = df_fe['Neighborhood'].values

for train_idx, val_idx in kf.split(df_fe):
    # Compute group means from train fold only
    train_df   = pd.DataFrame({'nbhd': nbhd_arr[train_idx], 'y': y_arr[train_idx]})
    fold_means = train_df.groupby('nbhd')['y'].mean()
    overall    = y_arr[train_idx].mean()
    # Apply to val fold (smoothed: blend with overall mean for small groups)
    for i in val_idx:
        nbhd = nbhd_arr[i]
        target_enc[i] = fold_means.get(nbhd, overall)

df_fe['Neighborhood_te'] = target_enc
r_correct = pd.Series(target_enc).corr(df_fe['log_SalePrice'])
print(f"   Cross-fold correlation with log(SalePrice): {r_correct:.4f}")
print()

# sklearn's TargetEncoder does the same correctly in a pipeline
print("   In a Pipeline, use sklearn's TargetEncoder:")
print("   TargetEncoder(target_type='continuous', smooth='auto')")
print("   -> Handles cross-fitting automatically when used in Pipeline + cross_val_score")
print()

# ── 4. Frequency encoding ─────────────────────────────────────────────────────
print("4. Frequency Encoding (how often does each neighborhood appear?)")
freq_map = df_fe['Neighborhood'].value_counts(normalize=True).to_dict()
df_fe['Neighborhood_freq'] = df_fe['Neighborhood'].map(freq_map)
print("   Neighborhood frequencies:")
for nbhd, freq in sorted(freq_map.items(), key=lambda x: -x[1]):
    print(f"     {nbhd:12}: {freq:.3f}")
print()

# ── Summary: compare all encoding correlations ────────────────────────────────
print("Encoding correlation summary (with log(SalePrice)):")
enc_cols = {
    'One-hot (BldgType_TwnhsE)': 'BldgType_TwnhsE',
    'Ordinal (Condition)':        'Condition_ord',
    'Target enc (Neighborhood)':  'Neighborhood_te',
    'Freq enc (Neighborhood)':    'Neighborhood_freq',
}
for label, col in enc_cols.items():
    r = df_fe[col].corr(df_fe['log_SalePrice'])
    print(f"  {label:35}: r = {r:+.4f}")


---
## Section 4 — GroupBy Features (Neighborhood Statistics)

### The Idea

Beyond target encoding, you can create rich statistics per group:
- **Mean / median price** per neighborhood (target encoding)
- **Standard deviation** — how variable is this neighborhood?
- **Count** — how many transactions? (stability proxy)
- **Percentile rank** — where does this house rank within its neighborhood?

These features give the model **neighborhood context** for each house —
something raw features don't encode.

### The Leakage Rule (repeated for emphasis)

**Always compute group statistics from the training set only.**
Never use test set rows when computing neighborhood means.

In production: compute stats from train, join to test. Use `TargetEncoder`
in an sklearn Pipeline for automatic handling.


In [ ]:
print("=== GroupBy / Neighborhood Features ===")
print()

# Simulate proper train/test split
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df_fe, test_size=0.2, random_state=42)
train_df = train_df.copy()
test_df  = test_df.copy()

# ── Compute group statistics on TRAIN only ────────────────────────────────────
nbhd_stats = (train_df.groupby('Neighborhood')
              .agg(
                  nbhd_mean_logprice  = ('log_SalePrice', 'mean'),
                  nbhd_median_price   = ('SalePrice',     'median'),
                  nbhd_std_logprice   = ('log_SalePrice', 'std'),
                  nbhd_count          = ('SalePrice',     'count'),
                  nbhd_mean_qual      = ('OverallQual',   'mean'),
                  nbhd_mean_area      = ('GrLivArea',     'mean'),
              )
              .round(4))

print("Neighborhood group statistics (from train set):")
print(nbhd_stats.sort_values('nbhd_mean_logprice', ascending=False).round(2))
print()

# ── Merge back to train and test (join on Neighborhood) ──────────────────────
train_df = train_df.merge(nbhd_stats, on='Neighborhood', how='left')
test_df  = test_df.merge(nbhd_stats,  on='Neighborhood', how='left')
# For test neighborhoods not seen in train, fill with global mean
global_means = {col: train_df[col].mean() for col in nbhd_stats.columns}
for col, val in global_means.items():
    test_df[col] = test_df[col].fillna(val)

print(f"Train after merge: {train_df.shape}")
print(f"Test  after merge: {test_df.shape}")
print()

# ── Within-neighborhood percentile rank ───────────────────────────────────────
# Where does this house rank within its neighborhood?
# Valuable for tree models — a "cheap NridgHt house" vs "expensive Edwards house"
train_df['nbhd_price_rank'] = (train_df.groupby('Neighborhood')['SalePrice']
                                .rank(pct=True))
# For test: rank against the neighborhood distribution from train
nbhd_prices = train_df.groupby('Neighborhood')['SalePrice'].apply(list).to_dict()

def rank_against_train(row):
    nbhd = row['Neighborhood']
    if nbhd not in nbhd_prices:
        return 0.5
    train_prices = nbhd_prices[nbhd]
    below = sum(1 for p in train_prices if p < row['SalePrice'])
    return below / len(train_prices)

test_df['nbhd_price_rank'] = test_df.apply(rank_against_train, axis=1)

print("Within-neighborhood price rank (sample):")
sample = train_df[['Neighborhood','SalePrice','nbhd_median_price','nbhd_price_rank']].head(8)
print(sample.round(2).to_string())
print()

# ── Measure lift from group features ──────────────────────────────────────────
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

base_cols  = ['GrLivArea','OverallQual','HouseAge','GarageCars','TotalBsmtSF',
              'area_x_qual','Condition_ord']
group_cols = ['nbhd_mean_logprice','nbhd_std_logprice','nbhd_mean_qual',
              'nbhd_mean_area','nbhd_price_rank']

train_base  = train_df[base_cols].values
train_full  = train_df[base_cols + group_cols].values
y_train_arr = train_df['log_SalePrice'].values

def cv_r2_arr(X, y):
    sc = StandardScaler()
    X_sc = sc.fit_transform(X)
    return cross_val_score(Ridge(alpha=10), X_sc, y, cv=5, scoring='r2').mean()

r2_base  = cv_r2_arr(train_base, y_train_arr)
r2_full  = cv_r2_arr(train_full, y_train_arr)
print(f"CV R2 without group features: {r2_base:.4f}")
print(f"CV R2 with    group features: {r2_full:.4f}  (+{r2_full-r2_base:.4f})")
print()
print("Group features are often top importances in tree models too.")


---
## Section 5 — Domain Knowledge Features

### Engineering for the Problem Domain

General transforms (log, polynomial) apply to any dataset.
Domain features apply **what you know about houses** — and they often outperform
the general ones because they encode structure the model can't discover from raw numbers.

House price domain knowledge:
- Total finished area matters more than above-ground area alone
- Renovation matters: a 1960 house renovated in 2020 is different from an untouched 1960 house
- Bathroom-to-bedroom ratio is a proxy for luxury
- "Bang for the buck" — price per square foot signals value vs. neighborhood

These are the features that separate top Kaggle submissions from baseline.


In [ ]:
print("=== Domain Knowledge Features ===")
print()

df_domain = df_fe.copy()

# ── Total area features ───────────────────────────────────────────────────────
df_domain['TotalSF']       = df_domain['GrLivArea'] + df_domain['TotalBsmtSF']
df_domain['log_TotalSF']   = np.log1p(df_domain['TotalSF'])

# Bathroom coverage (luxury proxy)
# FullBath + half (assuming half baths ≈ 0.5)
df_domain['BathBedRatio']  = (df_domain['FullBath'] + 0.5) / (df_domain['TotRmsAbvGrd'] - df_domain['FullBath'] + 1)

# ── Renovation signal ─────────────────────────────────────────────────────────
# If we had YearRemodAdd (year of last remodel), we'd use that
# Proxy: is the house "effectively new"? (renovated within 15 years of sale)
# Here we use YearBuilt as a proxy for age
df_domain['EffectiveAge']  = df_domain['HouseAge'].clip(0, 60)   # cap at 60 — very old is very old
df_domain['IsRecent']      = (df_domain['HouseAge'] < 20).astype(int)

# ── Space efficiency ──────────────────────────────────────────────────────────
df_domain['AvgRoomSize']   = df_domain['GrLivArea'] / (df_domain['TotRmsAbvGrd'] + 1)
df_domain['BasementRatio'] = df_domain['TotalBsmtSF'] / (df_domain['TotalSF'] + 1)
df_domain['GarageScore']   = df_domain['GarageCars'] * df_domain['OverallQual']

# ── Premium score composite ───────────────────────────────────────────────────
# Intuition: a house is premium if it's large, high quality, and new
# Normalized so each term is on similar scale
area_norm  = (df_domain['GrLivArea']  - df_domain['GrLivArea'].mean())  / df_domain['GrLivArea'].std()
qual_norm  = (df_domain['OverallQual']- df_domain['OverallQual'].mean()) / df_domain['OverallQual'].std()
age_norm   = -(df_domain['HouseAge']  - df_domain['HouseAge'].mean())   / df_domain['HouseAge'].std()
df_domain['PremiumScore'] = (area_norm + qual_norm * 1.5 + age_norm) / 3.5

print("New domain features:")
domain_feats = ['TotalSF','log_TotalSF','BathBedRatio','EffectiveAge',
                'AvgRoomSize','BasementRatio','GarageScore','PremiumScore']
for feat in domain_feats:
    r = df_domain[feat].corr(df_domain['log_SalePrice'])
    print(f"  {feat:18}: corr with log(SalePrice) = {r:+.4f}")
print()

# ── Feature importance using a quick tree model ───────────────────────────────
from sklearn.ensemble import RandomForestRegressor

all_feats = (['GrLivArea','OverallQual','HouseAge','GarageCars','TotalBsmtSF',
              'LotArea','TotRmsAbvGrd','FullBath','Condition_ord'] +
             ['area_x_qual','total_sf','avg_room_size','has_garage',
              'has_basement','is_new','is_high_qual'] +
             ['log_TotalSF','BathBedRatio','EffectiveAge','AvgRoomSize',
              'BasementRatio','GarageScore','PremiumScore'])

# Keep only features we have in df_domain
available = [f for f in all_feats if f in df_domain.columns]
X_all = df_domain[available].values
y_all = df_domain['log_SalePrice'].values

rf = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_all, y_all)
importances = pd.Series(rf.feature_importances_, index=available).sort_values(ascending=False)

print("Random Forest feature importances (top 15):")
fig, ax = plt.subplots(figsize=(10, 6))
top15 = importances.head(15)
colors = ['steelblue' if f in domain_feats or f in ['area_x_qual','total_sf','avg_room_size',
          'has_garage','has_basement','is_new','is_high_qual'] else 'coral' for f in top15.index]
ax.barh(top15.index[::-1], top15.values[::-1], color=colors[::-1], edgecolor='white')
ax.set_title("Feature Importances  (blue=engineered, red=original)", fontweight='bold')
ax.set_xlabel('importance')
# Legend
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='steelblue', label='Engineered'),
                   Patch(color='coral',     label='Original')], fontsize=9)
plt.tight_layout()
plt.savefig('/tmp/fe_importance.png', dpi=100, bbox_inches='tight')
plt.show()

print()
for feat, imp in top15.items():
    tag = " (engineered)" if feat in domain_feats + ['area_x_qual','total_sf','avg_room_size','PremiumScore'] else ""
    print(f"  {feat:20}: {imp:.4f}{tag}")


---
## Section 6 — Feature Selection

### Why Select Features?

More features is not always better:
- **Curse of dimensionality** — too many features, too few samples
- **Noise features** hurt linear models and slow training
- **Correlated features** cause multicollinearity in linear models
- **Interpretability** — fewer features = easier to debug

### Three Selection Methods

| Method | How | When |
|---|---|---|
| **Variance threshold** | Drop near-zero variance features | Always — quick first pass |
| **Correlation filter** | Drop redundant features (|r| > threshold) | Linear models |
| **Model-based** | Use tree importances or Lasso coefficients | All models |

These are applied **after** feature engineering — select from the full engineered set.


In [ ]:
from sklearn.feature_selection import VarianceThreshold, SelectFromModel
from sklearn.linear_model import LassoCV

print("=== Feature Selection ===")
print()

# Build the engineered feature set
feat_cols = ['GrLivArea','OverallQual','HouseAge','GarageCars','TotalBsmtSF',
             'LotArea','TotRmsAbvGrd','FullBath',
             'log_LotArea','log_TotalBsmtSF',
             'OverallQual_sq','HouseAge_sq',
             'area_x_qual','bsmt_x_qual','garage_x_qual',
             'avg_room_size','total_sf','garage_ratio',
             'has_garage','has_basement','is_new','is_large','is_high_qual',
             'Condition_ord','Neighborhood_te','Neighborhood_freq',
             'BldgType_TwnhsE']

# Add domain features from df_domain
domain_extra = ['log_TotalSF','BathBedRatio','EffectiveAge','AvgRoomSize',
                'BasementRatio','GarageScore','PremiumScore']
for col in domain_extra:
    if col not in df_domain.columns:
        continue
    feat_cols.append(col)

available = [c for c in feat_cols if c in df_domain.columns]
df_sel    = df_domain[available].copy()
y_sel     = df_domain['log_SalePrice'].values

print(f"Starting feature count: {len(available)}")
print()

# ── Step 1: Variance threshold ────────────────────────────────────────────────
print("Step 1: Variance Threshold (drop near-zero variance)")
selector_var = VarianceThreshold(threshold=0.01)
selector_var.fit(df_sel.values)
kept_var     = [c for c, ok in zip(available, selector_var.get_support()) if ok]
dropped_var  = [c for c, ok in zip(available, selector_var.get_support()) if not ok]
print(f"  Dropped {len(dropped_var)} features: {dropped_var}")
print(f"  Remaining: {len(kept_var)}")
print()

# ── Step 2: Correlation filter ────────────────────────────────────────────────
print("Step 2: Correlation Filter (drop if |r| > 0.90 with another feature)")
corr_matrix = df_sel[kept_var].corr().abs()
upper       = corr_matrix.where(np.triu(np.ones_like(corr_matrix, dtype=bool), k=1))
to_drop_corr= [col for col in upper.columns if any(upper[col] > 0.90)]
kept_corr   = [c for c in kept_var if c not in to_drop_corr]
print(f"  Dropped {len(to_drop_corr)} features: {to_drop_corr}")
print(f"  Remaining: {len(kept_corr)}")
print()

# ── Step 3: Lasso feature selection ──────────────────────────────────────────
print("Step 3: Lasso (L1 penalty drives weak feature coefficients to zero)")
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

X_corr = df_sel[kept_corr].values
sc     = StandardScaler()
X_sc   = sc.fit_transform(X_corr)

lasso    = LassoCV(cv=5, random_state=42, max_iter=3000)
lasso.fit(X_sc, y_sel)

lasso_coefs = pd.Series(np.abs(lasso.coef_), index=kept_corr)
kept_lasso  = lasso_coefs[lasso_coefs > 0].index.tolist()
dropped_l   = lasso_coefs[lasso_coefs == 0].index.tolist()
print(f"  Best alpha: {lasso.alpha_:.5f}")
print(f"  Dropped {len(dropped_l)} features with zero coefficient: {dropped_l}")
print(f"  Remaining: {len(kept_lasso)}")
print()

# ── Compare: baseline vs selected set ────────────────────────────────────────
from sklearn.linear_model import Ridge

def eval_feats(cols, label):
    X = df_sel[cols].values
    sc = StandardScaler()
    X_sc = sc.fit_transform(X)
    scores = cross_val_score(Ridge(alpha=10), X_sc, y_sel, cv=5, scoring='r2')
    print(f"  {label:40s}: {len(cols):2d} feats  CV R2 = {scores.mean():.4f} +/- {scores.std():.4f}")

print("Model comparison:")
eval_feats(available,   "All engineered features")
eval_feats(kept_var,    "After variance filter")
eval_feats(kept_corr,   "After correlation filter")
eval_feats(kept_lasso,  "After Lasso selection")
print()
print("Selected features:")
for feat in sorted(kept_lasso):
    coef = lasso_coefs[feat]
    print(f"  {feat:25}: |coef| = {coef:.4f}")


---
## Section 7 — The Complete Feature Engineering Pipeline

### Wrapping It All in sklearn

Everything built in this notebook can be encapsulated in a single
`Pipeline` + `ColumnTransformer` — the production-ready approach.

Benefits:
- No leakage — all statistics (means, target encodings) fit on train only
- No code duplication — apply the same transforms to train and test
- Works with `GridSearchCV` and `cross_val_score` automatically
- Can be saved and reloaded: `joblib.dump(pipeline, 'model.pkl')`

### The Full Pattern

```
ColumnTransformer
    numeric pipeline  -> [Impute -> clip -> log -> StandardScale]
    categorical pipeline -> [Impute -> encode]
          |
          v
    Selected features
          |
          v
    Ridge / Lasso / XGBoost
```


In [ ]:
from sklearn.pipeline import Pipeline, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, TargetEncoder
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import Ridge, LassoCV
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import r2_score
import warnings; warnings.filterwarnings('ignore')

print("=== The Complete Feature Engineering Pipeline ===")
print()

# ── Fresh start from raw data (no hand-made features) ────────────────────────
df_raw_cols = ['GrLivArea','OverallQual','YearBuilt','GarageCars','TotalBsmtSF',
               'LotArea','TotRmsAbvGrd','FullBath','Neighborhood','BldgType','Condition']
X_raw = df_clean[df_raw_cols].copy()
y_raw = df_clean['log_SalePrice'].values

# Train/test split FIRST
X_tr, X_te, y_tr, y_te = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42)
print(f"Train: {X_tr.shape}   Test: {X_te.shape}")
print()

# ── Define feature groups ─────────────────────────────────────────────────────
num_log   = ['LotArea','TotalBsmtSF']              # right-skewed -> log
num_poly  = ['OverallQual','YearBuilt']             # squared terms
num_std   = ['GrLivArea','GarageCars','TotRmsAbvGrd','FullBath']  # standard scale
cat_ord   = ['Condition']                           # ordinal: Poor < Fair < ... < Excellent
cat_target= ['Neighborhood']                        # target encode
cat_ohe   = ['BldgType']                           # one-hot

# ── Custom transformers ───────────────────────────────────────────────────────
log_transformer  = FunctionTransformer(np.log1p, inverse_func=np.expm1)

def add_poly_age(X):
    # X: [OverallQual, YearBuilt]  -> [OverallQual, HouseAge, OverallQual^2, HouseAge^2]
    qual     = X[:, 0]
    house_age= (2024 - X[:, 1]).clip(0, 80)
    return np.column_stack([qual, house_age, qual**2, house_age**2])

poly_transformer = FunctionTransformer(add_poly_age)

# ── Numeric sub-pipelines ─────────────────────────────────────────────────────
log_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('log',    log_transformer),
    ('scale',  StandardScaler()),
])
poly_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('poly',   poly_transformer),
    ('scale',  StandardScaler()),
])
std_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale',  StandardScaler()),
])

# ── Categorical sub-pipelines ─────────────────────────────────────────────────
condition_order = [['Poor','Fair','Typical','Good','Excellent']]
ord_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value='Typical')),
    ('encode', OrdinalEncoder(categories=condition_order,
                              handle_unknown='use_encoded_value', unknown_value=-1)),
    ('scale',  StandardScaler()),
])
target_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value='NAmes')),
    ('encode', TargetEncoder(target_type='continuous', smooth='auto', cv=5)),
    ('scale',  StandardScaler()),
])
ohe_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value='1Fam')),
    ('encode', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
])

# ── ColumnTransformer ─────────────────────────────────────────────────────────
preprocessor = ColumnTransformer([
    ('log',    log_pipe,    num_log),
    ('poly',   poly_pipe,   num_poly),
    ('std',    std_pipe,    num_std),
    ('ord',    ord_pipe,    cat_ord),
    ('target', target_pipe, cat_target),
    ('ohe',    ohe_pipe,    cat_ohe),
], remainder='drop')

# ── Full pipeline ─────────────────────────────────────────────────────────────
full_pipe = Pipeline([
    ('preprocess', preprocessor),
    ('model',      Ridge(alpha=10.0)),
])

# ── Cross-validation ──────────────────────────────────────────────────────────
cv_scores = cross_val_score(full_pipe, X_tr, y_tr, cv=5, scoring='r2')
full_pipe.fit(X_tr, y_tr)
test_r2   = full_pipe.score(X_te, y_te)
# Convert log R2 to approximate price R2
y_te_raw  = np.expm1(y_te)
pred_log  = full_pipe.predict(X_te)
pred_raw  = np.expm1(pred_log)

print("Pipeline Results:")
print(f"  CV R2 (5-fold): {cv_scores.round(3)}  mean={cv_scores.mean():.4f}  std={cv_scores.std():.4f}")
print(f"  Test R2 (log):  {test_r2:.4f}")
print(f"  Test R2 (price):{r2_score(y_te_raw, pred_raw):.4f}")
rmsle = np.sqrt(np.mean((np.log1p(y_te_raw) - np.log1p(pred_raw))**2))
print(f"  Test RMSLE:     {rmsle:.4f}")
print()

# ── Predict on brand-new houses ───────────────────────────────────────────────
print("Predicting on new houses:")
new_houses = pd.DataFrame({
    'GrLivArea':    [1400.0,  3200.0,   900.0],
    'OverallQual':  [6,       9,        4],
    'YearBuilt':    [1985,    2018,     1962],
    'GarageCars':   [1.0,     3.0,      0.0],
    'TotalBsmtSF':  [800.0,   2000.0,   0.0],
    'LotArea':      [8000.0,  12000.0,  5000.0],
    'TotRmsAbvGrd': [6,       10,       5],
    'FullBath':     [1,       3,        1],
    'Neighborhood': ['NAmes', 'NridgHt','OldTown'],
    'BldgType':     ['1Fam',  '1Fam',   'Duplex'],
    'Condition':    ['Typical','Excellent','Fair'],
})
log_preds  = full_pipe.predict(new_houses)
price_preds= np.expm1(log_preds)
for i, row in new_houses.iterrows():
    print(f"  House {i+1}: {row['Neighborhood']:10} qual={row['OverallQual']}  area={row['GrLivArea']:.0f}  -> ${price_preds[i]:,.0f}")
print()
print("Pipeline saved/loaded with: joblib.dump(full_pipe, 'house_pipeline.pkl')")


---
## Summary — Feature Engineering

### The Engineering Playbook

```python
# 1. NUMERIC TRANSFORMS
df['log_col']    = np.log1p(df['col'])           # right-skewed
df['col_clip']   = df['col'].clip(p01, p99)      # winsorize outliers
df['qual_sq']    = df['OverallQual'] ** 2        # non-linear relationship
df['age_sq']     = df['HouseAge'] ** 2

# 2. INTERACTIONS
df['area_x_qual']  = df['GrLivArea'] * df['OverallQual']
df['avg_room_size']= df['GrLivArea'] / (df['TotRmsAbvGrd'] + 1)
df['total_sf']     = df['GrLivArea'] + df['TotalBsmtSF']
df['has_garage']   = (df['GarageCars'] > 0).astype(int)

# 3. CATEGORICAL ENCODING
# One-hot (low cardinality)
pd.get_dummies(df['BldgType'], drop_first=True)
# Ordinal (natural order)
df['Condition_ord'] = df['Condition'].map({'Poor':1,...,'Excellent':5})
# Target (high cardinality — always cross-fit!)
TargetEncoder(target_type='continuous', cv=5)

# 4. GROUPBY FEATURES
nbhd_stats = train_df.groupby('Neighborhood')['target'].agg(['mean','std'])
df = df.merge(nbhd_stats, on='Neighborhood', how='left')

# 5. FEATURE SELECTION
VarianceThreshold(threshold=0.01)   # drop near-zero variance
LassoCV(cv=5)                        # L1 drives weak features to zero

# 6. THE PIPELINE
ColumnTransformer([
    ('num',    Pipeline([('impute',SimpleImputer()),('scale',StandardScaler())]),   num_cols),
    ('target', Pipeline([('encode',TargetEncoder())]),                              cat_cols),
]) |> Ridge / XGBoost
```

### The Rules

1. **Always predict log(SalePrice)** for price targets — exponentiate back after
2. **Compute group statistics from train only** — leakage is silent and deadly
3. **Use TargetEncoder inside a Pipeline** — it handles cross-fitting automatically
4. **Polynomial features before scaling** — scaling after doesn't change the polynomial
5. **Measure every feature's lift** — correlation and CV R2 before/after

### Next Steps
- `ml-algorithms/regression/regression_kaggle.ipynb` — put everything together on the real Kaggle dataset
- `classification/` — next module after regression
